# AI 그림 공방 🖌️ (낙서 실습 2회차)

**한국외국어대학교 인공지능 · 보너스 실습**

1회차에서는 낙서를 **알아보는** CNN을 만들었다. 2회차에서는 그림을 **만드는** AI를 다룬다.

| 단계 | 할 일 | 시간(T4 GPU) |
|---|---|---|
| 0 | 준비: Cloudflare 토큰 입력 | 2분 |
| 1 | 낙서 데이터와 심판 CNN 다시 만들기 (1회차 복습) | 2분 |
| 2 | **내 낙서가 작품으로**: 그림판에 그리면 CNN이 알아보고, 이미지 생성 AI가 수채화·점토 인형 등으로 다시 그린다 | 자유 |
| 3 | **AI 전화 게임**: 문장 → 그림 → 설명 → 그림 … 을 돌리면 처음 문장이 어떻게 바뀔까? | 5분 |
| 4 | **원하는 낙서 주문하기**: 조건부 확산 모델을 직접 학습시켜 "별 그려 줘"에 답하게 하기 | 8~12분 |

**시작 전에**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한다.
2. 2·3단계는 Cloudflare Workers AI(무료)를 쓴다. 아래 「Cloudflare 토큰 만들기」를 따라 Account ID와 API 토큰을 준비한다. 토큰이 없으면 0단계에서 Enter만 누르고 넘어가면 되고, 1·4단계는 토큰 없이 동작한다.

<details><summary><b>Cloudflare 토큰 만들기 (처음 한 번, 약 3분)</b></summary>

1. [dash.cloudflare.com](https://dash.cloudflare.com/sign-up)에서 무료 계정을 만들고 로그인한다.
2. 왼쪽 메뉴 `AI → Workers AI`로 들어가 `REST API 사용`을 누른다.
3. `Workers AI API 토큰 만들기`를 눌러 토큰을 만들고 복사해 둔다(다시 볼 수 없으니 메모장에 붙여 둔다).
4. 같은 화면에 보이는 **Account ID**도 복사한다.

무료 사용량은 하루 10,000 neurons다. 이 노트북 기준으로 FLUX.1 schnell 그림 한 장(1024×1024, 4스텝)이 약 173 neurons, FLUX.2 klein 변환 한 장(512×512)이 약 110 neurons다. 한도를 넘으면 다음 날(UTC 0시) 다시 채워진다.
</details>

그림 제목과 축 이름은 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시한다.

## 0. 준비

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio"], check=False)

import os, io, math, time, base64, random, textwrap, urllib.request, urllib.parse
from getpass import getpass
import numpy as np, requests
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(GPU 없음: 4단계가 느려진다)")

In [ ]:
ACC = os.environ.get("CLOUDFLARE_ACCOUNT_ID") or input("Cloudflare Account ID (없으면 Enter): ").strip()
TOK = os.environ.get("CLOUDFLARE_API_TOKEN") or (getpass("Cloudflare API 토큰 (없으면 Enter): ").strip() if ACC else "")
CF_OK = bool(ACC and TOK)
CF_BASE = f"https://api.cloudflare.com/client/v4/accounts/{ACC}/ai/run/"
USAGE = {"calls": 0, "neurons_est": 0.0}

class CFError(Exception): pass

def cf_post(model, json_body=None, files=None, est=0.0, retries=3):
    # Workers AI 호출. 성공하면 result를, 실패하면 이유를 담은 CFError를 낸다.
    if not CF_OK: raise CFError("Cloudflare 토큰이 없어 이 단계를 건너뛴다.")
    for k in range(retries):
        r = requests.post(CF_BASE + model, headers={"Authorization": f"Bearer {TOK}"},
                          json=json_body, files=files, timeout=180)
        try: d = r.json()
        except ValueError: raise CFError(f"HTTP {r.status_code}")
        if d.get("success"):
            USAGE["calls"] += 1; USAGE["neurons_est"] += est
            return d["result"]
        msg = "; ".join(e.get("message", "") for e in d.get("errors", []))
        if "flagged" in msg: raise CFError("안전 필터에 걸렸다. 스타일이나 문장을 바꿔 다시 시도한다.")
        if r.status_code in (429, 503) or "Capacity" in msg: time.sleep(3 * (k + 1)); continue
        raise CFError(f"HTTP {r.status_code}: {msg[:200]}")
    raise CFError("요청이 몰려 실패했다. 잠시 뒤 다시 실행한다.")

def text_to_image(prompt, steps=4):
    # FLUX.1 schnell: 글 → 그림 (1024×1024). 사용량 약 4타일 × (4.8 + 9.6 × steps) neurons
    res = cf_post("@cf/black-forest-labs/flux-1-schnell", {"prompt": prompt, "steps": steps}, est=4 * (4.8 + 9.6 * steps))
    return Image.open(io.BytesIO(base64.b64decode(res["image"]))).convert("RGB")

def image_edit(prompt, pil_img, size=512):
    # FLUX.2 klein: 입력 그림 + 지시문 → 새 그림
    b = io.BytesIO(); pil_img.convert("RGB").resize((size, size)).save(b, "PNG")
    files = {"prompt": (None, prompt), "input_image_0": ("in.png", b.getvalue(), "image/png"),
             "width": (None, str(size)), "height": (None, str(size))}
    res = cf_post("@cf/black-forest-labs/flux-2-klein-4b", files=files, est=110)
    return Image.open(io.BytesIO(base64.b64decode(res["image"]))).convert("RGB")

def describe(pil_img):
    # LLaVA 1.5: 그림 → 한 문장 설명 (영어)
    b = io.BytesIO(); pil_img.convert("RGB").resize((512, 512)).save(b, "JPEG", quality=90)
    res = cf_post("@cf/llava-hf/llava-1.5-7b-hf",
                  {"image": list(b.getvalue()), "prompt": "Describe this picture in one short sentence.", "max_tokens": 60})
    return res["description"].strip()

print("Cloudflare 연결:", "사용" if CF_OK else "없음 (2·3단계 건너뜀)")

## 1. 낙서 데이터와 심판 CNN (1회차 복습)

1회차와 같은 낙서 10종류를 받고, 같은 구조의 CNN을 빠르게 다시 학습시킨다. 이 CNN은 2단계에서 내 낙서가 무엇인지 알아보고, 4단계에서 확산 모델이 그린 낙서를 채점한다.

In [ ]:
CATEGORIES = ["apple", "bicycle", "cat", "donut", "fish", "house", "smiley face", "star", "tree", "umbrella"]
KOREAN = {"apple": "사과", "bicycle": "자전거", "cat": "고양이", "donut": "도넛", "fish": "물고기",
          "house": "집", "smiley face": "웃는 얼굴", "star": "별", "tree": "나무", "umbrella": "우산"}
N_PER_CLASS = 6000
QD = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

def fetch_doodles(name, n):
    url = QD + urllib.parse.quote(name) + ".npy"
    head = urllib.request.urlopen(urllib.request.Request(url, headers={"Range": "bytes=0-1023"})).read()
    f = io.BytesIO(head); major, _ = np.lib.format.read_magic(f)
    reader = np.lib.format.read_array_header_1_0 if major == 1 else np.lib.format.read_array_header_2_0
    shape, _, _ = reader(f); start, n = f.tell(), min(n, shape[0])
    body = urllib.request.urlopen(urllib.request.Request(url, headers={"Range": f"bytes={start}-{start + n*784 - 1}"})).read()
    return np.frombuffer(body, dtype=np.uint8).reshape(n, 28, 28)

t0 = time.time()
X_all = np.concatenate([fetch_doodles(c, N_PER_CLASS) for c in CATEGORIES])
y_all = np.repeat(np.arange(len(CATEGORIES)), N_PER_CLASS)
print(f"낙서 {X_all.shape} · {time.time()-t0:.1f}초")

perm = np.random.RandomState(0).permutation(len(X_all))
is_test = np.zeros(len(X_all), bool); is_test[perm[:5000]] = True
Xtr = torch.tensor(X_all[~is_test], dtype=torch.float32).unsqueeze(1).div(255).to(device); ytr = torch.tensor(y_all[~is_test]).to(device)
Xte = torch.tensor(X_all[is_test], dtype=torch.float32).unsqueeze(1).div(255).to(device); yte = torch.tensor(y_all[is_test]).to(device)

judge = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CATEGORIES))).to(device)
opt = torch.optim.Adam(judge.parameters(), 1e-3)
for ep in range(3):
    judge.train(); order = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), 128):
        b = order[i:i+128]; loss = F.cross_entropy(judge(Xtr[b]), ytr[b])
        opt.zero_grad(); loss.backward(); opt.step()
judge.eval()
with torch.no_grad(): acc = (judge(Xte).argmax(1) == yte).float().mean().item()
print(f"심판 CNN 평가 정확도: {acc:.1%}")

@torch.no_grad()
def judge_probs(imgs_uint8):
    x = torch.tensor(np.asarray(imgs_uint8), dtype=torch.float32).view(-1, 1, 28, 28).div(255).to(device)
    return F.softmax(judge(x), 1).cpu().numpy()

## 2. 내 낙서가 작품으로 🎨

흐름은 세 단계다.

1. **CNN이 알아보기**: 낙서를 28×28로 줄여 심판 CNN에 넣고 무엇인지 맞힌다.
2. **지시문 만들기**: "Turn this rough doodle of a *cat* into a *watercolor illustration*…"처럼 CNN의 답과 고른 스타일로 문장을 만든다.
3. **이미지 생성 AI가 다시 그리기**: Cloudflare Workers AI의 FLUX.2 klein이 원래 낙서의 구도를 살려 새 그림을 만든다.

먼저 데이터셋의 낙서 세 장으로 시험한다.

In [ ]:
STYLES = {
    "수채화": "a soft watercolor illustration",
    "크레용 동화책": "a children's picture book page drawn with crayons",
    "점토 인형": "a cute 3D clay figure photographed on a table",
    "유화": "an oil painting with thick brush strokes",
    "종이 공예": "a colorful layered paper craft",
    "스테인드글라스": "a stained glass window",
}

def doodle_prompt(label, style_key, extra=""):
    p = f"Turn this rough doodle of a {label} into {STYLES[style_key]}. Keep the same pose and composition."
    return p + (" " + extra.strip() if extra.strip() else "")

def doodle_to_png(img28_or_big):
    # 흰 배경 검은 선으로 바꾸고 512×512로 키운다 (생성 AI 입력용)
    a = np.asarray(img28_or_big).astype(np.float32)
    if a.max() > 0 and a.mean() < 127: a = 255 - a        # 검은 배경 흰 선 → 흰 배경 검은 선
    return Image.fromarray(a.astype(np.uint8)).convert("RGB").resize((512, 512), Image.BICUBIC)

demo_idx = [np.where(y_all == CATEGORIES.index(c))[0][5] for c in ["cat", "house", "fish"]]
demo_styles = ["수채화", "점토 인형", "크레용 동화책"]
if CF_OK:
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    for c, (i, sk) in enumerate(zip(demo_idx, demo_styles)):
        p = judge_probs(X_all[i][None])[0]; label = CATEGORIES[p.argmax()]
        axes[0, c].imshow(X_all[i], cmap="gray"); axes[0, c].set_title(f"CNN: {label} ({p.max():.0%})"); axes[0, c].axis("off")
        try:
            out = image_edit(doodle_prompt(label, sk), doodle_to_png(X_all[i]))
            axes[1, c].imshow(out)
        except CFError as e:
            print("실패:", e)
        axes[1, c].set_title("\n".join(textwrap.wrap(STYLES[sk], 30)), fontsize=9); axes[1, c].axis("off")
    plt.tight_layout(); plt.show()
    print(f"지금까지 호출 {USAGE['calls']}회 · 추정 사용량 약 {USAGE['neurons_est']:.0f} neurons")
else:
    print("Cloudflare 토큰이 없어 건너뛴다.")

### 🖍️ 그림판에서 직접 해 보기

아래 셀을 실행하면 그림판이 나온다. 10가지(사과, 자전거, 고양이, 도넛, 물고기, 집, 웃는 얼굴, 별, 나무, 우산) 중 하나를 그리고 스타일을 고른 뒤 **[그리기]**를 누른다.

- CNN이 엉뚱하게 알아보면 그 답대로 그림이 나온다. 일부러 헷갈리게 그려 보자.
- "추가 요청"에 `wearing a party hat`, `at night under the stars`처럼 영어로 덧붙이면 반영된다.
- 한 번 그릴 때마다 약 110 neurons를 쓴다(하루 무료 10,000).

In [ ]:
def sketch_to_arrays(img):
    # 그림판 이미지 → (CNN용 28×28 흰 선/검은 배경, 생성 AI용 512×512 흰 배경/검은 선)
    if isinstance(img, dict): img = img.get("composite", img.get("image"))
    a = np.asarray(img).astype(np.float32)
    if a.ndim == 3 and a.shape[2] == 4: ink = a[..., 3] * (255 - a[..., :3].mean(2)) / 255
    elif a.ndim == 3: ink = 255 - a[..., :3].mean(2)
    else: ink = 255 - a if a.mean() > 127 else a
    ys, xs = np.where(ink > 30)
    if len(ys) == 0: return None, None
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    s = max(y1 - y0, x1 - x0) + 1; pad = int(s * 0.12)
    canvas = np.zeros((s + 2 * pad, s + 2 * pad), np.float32)
    oy, ox = pad + (s - (y1 - y0 + 1)) // 2, pad + (s - (x1 - x0 + 1)) // 2
    canvas[oy:oy + y1 - y0 + 1, ox:ox + x1 - x0 + 1] = ink[y0:y1 + 1, x0:x1 + 1]
    small = F.interpolate(torch.tensor(canvas)[None, None], size=(28, 28), mode="area")[0, 0].numpy()
    d28 = np.clip(small / max(small.max(), 1e-6) * 255, 0, 255).astype(np.uint8)
    big = Image.fromarray((255 - np.clip(canvas, 0, 255)).astype(np.uint8)).convert("RGB").resize((512, 512))
    return d28, big

def studio(img, style_key, extra):
    d28, big = sketch_to_arrays(img)
    if d28 is None: return {"그림을 그려 주세요": 1.0}, None, ""
    p = judge_probs(d28[None])[0]; label = CATEGORIES[p.argmax()]
    guess = {f"{KOREAN[c]} ({c})": float(p[k]) for k, c in enumerate(CATEGORIES)}
    prompt = doodle_prompt(label, style_key, extra)
    try:
        out = image_edit(prompt, big)
        return guess, out, f"지시문: {prompt}\n추정 누적 사용량: 약 {USAGE['neurons_est']:.0f} neurons"
    except CFError as e:
        return guess, None, f"실패: {e}"

import gradio as gr
with gr.Blocks(title="AI 그림 공방") as app:
    gr.Markdown("### AI 그림 공방: 낙서를 그리고 스타일을 고르면 AI가 다시 그린다")
    with gr.Row():
        with gr.Column():
            try:
                pad = gr.Sketchpad(canvas_size=(360, 360), type="numpy",
                                   brush=gr.Brush(default_size=10, colors=["#000000"], color_mode="fixed"))
            except Exception:
                pad = gr.Sketchpad(type="numpy")
            style = gr.Dropdown(list(STYLES), value="수채화", label="스타일")
            extra = gr.Textbox(label="추가 요청 (영어, 선택)", placeholder="wearing a party hat")
            btn = gr.Button("그리기", variant="primary")
        with gr.Column():
            guess_out = gr.Label(num_top_classes=3, label="CNN이 알아본 것")
            img_out = gr.Image(label="AI가 다시 그린 그림")
            log_out = gr.Textbox(label="기록", lines=3)
    btn.click(studio, [pad, style, extra], [guess_out, img_out, log_out])
app.launch(debug=False)

## 3. AI 전화 게임 📞

말 전달 게임처럼, 한 문장을 AI 두 개가 번갈아 전달한다.

```
문장 → (FLUX.1 schnell) 그림 → (LLaVA) 한 문장 설명 → 그림 → 설명 → …
```

그림을 그리는 AI와 그림을 설명하는 AI가 서로 다른 모델이라, 전달될 때마다 무엇이 남고 무엇이 사라지는지 보인다. `START`를 바꿔 팀별로 해 보고, 가장 많이 바뀐 결과를 뽑아 보자.

> 한 판(4라운드)에 그림 4장, 약 700~800 neurons를 쓴다. 문장은 영어로 쓴다.

In [ ]:
START = "a penguin eating ice cream on a sunny beach"   # ← 바꿔 보자
ROUNDS = 4

def telephone(start, rounds=4):
    chain, text = [], start
    for r in range(rounds):
        img = text_to_image(text)
        cap = describe(img)
        chain.append((text, img, cap)); text = cap
        print(f"[{r+1}] 그림의 원문: {chain[-1][0]}\n    LLaVA 설명: {cap}")
    return chain

if CF_OK:
    try:
        chain = telephone(START, ROUNDS)
        fig, axes = plt.subplots(1, len(chain), figsize=(4 * len(chain), 5))
        for ax, (t, im, cap) in zip(np.atleast_1d(axes), chain):
            ax.imshow(im); ax.axis("off")
            ax.set_title("\n".join(textwrap.wrap(t, 30)), fontsize=11)
        plt.suptitle("AI telephone: prompt -> image -> caption -> image ...", fontsize=11); plt.tight_layout(); plt.show()
        print(f"\n처음: {START}\n마지막 설명: {chain[-1][2]}")
        print(f"추정 누적 사용량: 약 {USAGE['neurons_est']:.0f} neurons (LLaVA 제외)")
    except CFError as e:
        print("실패:", e)
else:
    print("Cloudflare 토큰이 없어 건너뛴다.")

> 🤔 몇 라운드 만에 처음 문장의 어떤 정보가 사라졌는가? 색, 개수, 장소, 행동 중 무엇이 가장 먼저 사라지는가? 설명 AI에게 "Describe this picture in detail"처럼 더 길게 요청하면(`describe` 함수의 prompt) 결과가 어떻게 달라질까?

## 4. 원하는 낙서 주문하기: 조건부 확산 모델 🎯

2·3단계는 남이 학습시킨 거대한 모델을 빌려 썼다. 이번에는 **작은 확산 모델을 직접 학습**시킨다. 1회차 보너스의 확산 모델은 한 종류(웃는 얼굴)만 그렸지만, 이번 모델은 "별", "고양이"처럼 **주문받은 종류**를 그린다.

방법은 두 가지를 더한다.

- **조건 넣기**: 잡음 예측 신경망 ε(x<sub>t</sub>, t, **y**)에 카테고리 y를 함께 넣는다.
- **가이던스(classifier-free guidance)**: 학습할 때 10%는 카테고리를 가리고("빈칸") 학습한다. 생성할 때는 조건 있는 예측과 없는 예측의 차이를 w배로 키운다.

  ε̂ = ε(x<sub>t</sub>, t, 빈칸) + w · (ε(x<sub>t</sub>, t, y) − ε(x<sub>t</sub>, t, 빈칸))

  w = 0이면 카테고리를 무시하고, w = 1이면 보통의 조건부 생성, w가 클수록 주문한 카테고리를 더 강하게 따른다. Stable Diffusion 같은 이미지 생성 AI의 `guidance scale` 설정이 바로 이것이다.

In [ ]:
GEN_CLASSES = ["smiley face", "star", "cat", "house"]    # 주문 가능한 낙서
NULL = len(GEN_CLASSES)                                    # "빈칸" 카테고리 번호
T = 400
betas = torch.linspace(1e-4, 0.02, T, device=device); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)

sel = np.isin(y_all, [CATEGORIES.index(c) for c in GEN_CLASSES])
gdata = torch.tensor(X_all[sel], dtype=torch.float32, device=device).unsqueeze(1) / 127.5 - 1
gdata = F.pad(gdata, (2, 2, 2, 2), value=-1)                                     # 32×32, [-1, 1]
glabel = torch.tensor([GEN_CLASSES.index(CATEGORIES[k]) for k in y_all[sel]], device=device)
print("학습 낙서:", tuple(gdata.shape), "· 카테고리:", GEN_CLASSES)

class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 2), nn.SiLU(), nn.Linear(dim * 2, dim * 2))
    def forward(self, t):
        half = self.dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        e = t.float()[:, None] * f[None]
        return self.mlp(torch.cat([e.sin(), e.cos()], 1))

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1); self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.n1 = nn.GroupNorm(8, cout); self.n2 = nn.GroupNorm(8, cout); self.t = nn.Linear(tdim, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, e):
        h = F.silu(self.n1(self.c1(x))) + self.t(e)[:, :, None, None]
        return F.silu(self.n2(self.c2(h))) + self.skip(x)

class CondUNet(nn.Module):
    def __init__(self, n_cls, c=64, tdim=128):
        super().__init__()
        self.te = TimeEmb(tdim // 2); self.ce = nn.Embedding(n_cls + 1, tdim)    # +1: 빈칸
        self.inc = nn.Conv2d(1, c, 3, padding=1)
        self.d1 = Block(c, c, tdim); self.d2 = Block(c, 2 * c, tdim); self.mid = Block(2 * c, 2 * c, tdim)
        self.u2 = Block(4 * c, c, tdim); self.u1 = Block(2 * c, c, tdim); self.out = nn.Conv2d(c, 1, 3, padding=1)
    def forward(self, x, t, y):
        e = self.te(t) + self.ce(y)                          # 시간과 카테고리 정보를 더해 모든 블록에 넣는다
        h0 = self.d1(self.inc(x), e); h1 = self.d2(F.avg_pool2d(h0, 2), e); m = self.mid(F.avg_pool2d(h1, 2), e)
        u = self.u2(torch.cat([F.interpolate(m, scale_factor=2), h1], 1), e)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h0], 1), e)
        return self.out(u)

torch.manual_seed(SEED)
net = CondUNet(len(GEN_CLASSES)).to(device)
STEPS = 5000 if device == "cuda" else 800
opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=STEPS)
print("파라미터 수:", f"{sum(p.numel() for p in net.parameters()):,}", "· 학습 스텝:", STEPS)

t0 = time.time(); run = None
for step in range(1, STEPS + 1):
    idx = torch.randint(0, len(gdata), (128,), device=device)
    x0, y = gdata[idx], glabel[idx].clone()
    y[torch.rand(len(y), device=device) < 0.1] = NULL          # 10%는 카테고리를 가리고 학습
    t = torch.randint(0, T, (len(x0),), device=device); eps = torch.randn_like(x0)
    xt = abar[t].sqrt()[:, None, None, None] * x0 + (1 - abar[t]).sqrt()[:, None, None, None] * eps
    loss = F.mse_loss(net(xt, t, y), eps)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    run = loss.item() if run is None else 0.98 * run + 0.02 * loss.item()
    if step % 1000 == 0 or step == STEPS:
        print(f"step {step:5d} · 잡음 예측 손실 {run:.4f} · {time.time()-t0:.0f}초")

In [ ]:
@torch.no_grad()
def order(cls_name, n=8, w=3.0, seed=0):
    # 카테고리 cls_name의 낙서 n장을 가이던스 세기 w로 생성한다.
    net.eval(); g = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 1, 32, 32, device=device, generator=g)
    yc = torch.full((n,), GEN_CLASSES.index(cls_name), device=device); yn = torch.full((n,), NULL, device=device)
    for t in range(T - 1, -1, -1):
        tt = torch.full((n,), t, device=device, dtype=torch.long)
        e = net(torch.cat([x, x]), torch.cat([tt, tt]), torch.cat([yc, yn]))
        e_c, e_u = e[:n], e[n:]
        e = e_u + w * (e_c - e_u)                                 # classifier-free guidance
        x = (x - betas[t] / (1 - abar[t]).sqrt() * e) / alphas[t].sqrt()
        if t > 0: x = x + betas[t].sqrt() * torch.randn(x.shape, device=device, generator=g)
    return ((x.clamp(-1, 1)[:, 0, 2:30, 2:30] + 1) * 127.5).cpu().numpy().astype(np.uint8)

W_LIST = [0.0, 1.0, 3.0]
fig, axes = plt.subplots(len(GEN_CLASSES), len(W_LIST), figsize=(11, 2.3 * len(GEN_CLASSES)))
results = {}
for r, c in enumerate(GEN_CLASSES):
    for k, w in enumerate(W_LIST):
        imgs = order(c, n=16, w=w, seed=r)
        results[(c, w)] = imgs
        strip = np.concatenate([np.concatenate(list(imgs[i:i+8]), 1) for i in (0, 8)], 0)
        axes[r, k].imshow(strip, cmap="gray"); axes[r, k].axis("off")
        if r == 0: axes[r, k].set_title(f"guidance w = {w}", fontsize=11)
    axes[r, 0].text(-8, 28, c, ha="right", va="center", fontsize=11)
plt.suptitle("Ordered doodles: rows = requested class, columns = guidance scale", fontsize=11)
plt.tight_layout(); plt.show()

심판 CNN에게 채점을 맡긴다. **주문 적중률**은 CNN이 주문한 카테고리로 알아본 비율이고, **다양성**은 같은 주문으로 나온 16장끼리 픽셀 차이의 평균이다(클수록 서로 다르게 그렸다).

In [ ]:
def diversity(imgs):
    v = imgs.reshape(len(imgs), -1).astype(np.float32) / 255
    d = ((v[:, None] - v[None]) ** 2).mean(-1)
    return d[np.triu_indices(len(v), 1)].mean()

print(f"{'주문':12s}" + "".join(f"{'w='+str(w):>18s}" for w in W_LIST))
print(f"{'':12s}" + "".join(f"{'적중률 / 다양성':>18s}" for _ in W_LIST))
for c in GEN_CLASSES:
    row = f"{KOREAN[c]:10s}"
    for w in W_LIST:
        imgs = results[(c, w)]; hit = (judge_probs(imgs).argmax(1) == CATEGORIES.index(c)).mean()
        row += f"{hit:>11.0%} / {diversity(imgs):.3f}"
    print(row)

> 🤔 w를 키우면 적중률과 다양성은 각각 어떻게 변했는가? 이미지 생성 서비스에서 guidance scale을 너무 크게 두면 어떤 그림이 나올지 예상해 보자.

직접 주문해 보자. 카테고리, 장수, 가이던스 세기를 바꿔 실행한다.

In [ ]:
MY_ORDER, N, W = "star", 12, 3.0          # ← "smiley face", "star", "cat", "house" 중 하나
imgs = order(MY_ORDER, n=N, w=W, seed=random.randint(0, 9999))
p = judge_probs(imgs)
fig, axes = plt.subplots(1, N, figsize=(N * 1.1, 1.6))
for ax, im, pr in zip(axes, imgs, p):
    ax.imshow(im, cmap="gray"); ax.axis("off"); ax.set_title(CATEGORIES[pr.argmax()][:7], fontsize=7)
plt.suptitle(f"order: {MY_ORDER}, w={W} (title = judge CNN's answer)", fontsize=10); plt.tight_layout(); plt.show()

## 5. 도전 과제

1. **스타일 대회**: 2단계 그림판에서 같은 낙서로 스타일 6개를 모두 만들어 가장 마음에 드는 것을 고른다. 추가 요청 문구로 이야기를 붙여 본다.
2. **CNN 속이기 + 생성**: CNN이 틀리게 알아보도록 낙서를 그리면 생성 AI는 무엇을 그리는가? 사람의 의도와 CNN의 판단 중 무엇이 결과를 정하는지 설명한다.
3. **전화 게임 분석**: 서로 다른 시작 문장 3개로 4라운드씩 돌려, 가장 먼저 사라지는 정보의 종류(색, 개수, 장소, 행동)를 표로 정리한다.
4. **가이던스 탐구**: `W_LIST`에 6.0, 10.0을 더해 적중률과 다양성의 변화를 그래프로 그린다.
5. **주문 메뉴 늘리기**: `GEN_CLASSES`에 1회차의 다른 카테고리를 넣어 다시 학습한다. 카테고리가 늘면 같은 학습 스텝에서 품질이 어떻게 달라지는가?

*데이터: Google Creative Lab, The Quick, Draw! Dataset (CC BY 4.0). 이미지 생성: Cloudflare Workers AI (FLUX.1 schnell, FLUX.2 klein 4B, LLaVA 1.5 7B).*